# monomech + GATMA exact-model workflow

This notebook shows two ways to use `monomech`:

1. **modular stage-by-stage execution** with DataFrame inspection after every step
2. **full wrapper execution** using the GATMA-specific OpenSim preset

This version is aligned to:
- MediaPipe `pose2d`
- MediaPipe `world3d` (root-centered 3D)
- `pnp` from `pose2d + world3d`
- `global_pose` from `pose2d + world3d + pnp`
- GATMA marker/body exports for OpenSim


In [ ]:
import monomech as mm
import pandas as pd
from pathlib import Path


## 1) Set your paths

Replace these with your own files before running the notebook.


In [ ]:
VIDEO_PATH = Path(r"subject01.mp4")
MODEL_TASK_PATH = Path(r"pose_landmarker_heavy.task")
OPENSIM_MODEL_PATH = Path(r"GATMA_Model.osim")
OUTPUT_DIR = Path("outputs") / VIDEO_PATH.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2) Build the trial

In [ ]:
trial = mm.Trial.from_video(VIDEO_PATH)
trial.video_metadata


## 3) Run each stage explicitly

In [ ]:
config = mm.PipelineConfig(
    pose=mm.MediaPipePoseConfig(
        model_asset_path=str(MODEL_TASK_PATH),
        target_fps=60,
        stride=1,
    ),
    opensim=mm.OpenSimConfig(
        model_path=str(OPENSIM_MODEL_PATH),
        model_preset="gatma_exact",
        run_scale=True,
        run_ik=True,
        run_id=False,
        export_marker_trc=True,
        export_landmark_trc=True,
        write_csv_copies=True,
    ),
)

pose2d = mm.pose2d.process(trial, config=config.pose)
world3d = mm.world3d.process(trial, pose2d=pose2d, config=config.pose)
pnp = mm.pnp.solve(trial, pose2d=pose2d, world3d=world3d, config=config.pnp)
global_pose = mm.global_pose.estimate(
    trial,
    pose2d=pose2d,
    world3d=world3d,
    pnp=pnp,
    config=config.global_pose,
)


## 4) Inspect DataFrames after each stage

In [ ]:
pose2d.tables["landmarks_long"].head()


In [ ]:
world3d.tables["world3d_long"].head()


In [ ]:
pnp.tables["camera_pose"].head()


In [ ]:
global_pose.tables["global_pose_long"].head()


In [ ]:
global_pose.tables["contacts"].head()


## 5) Notebook visualizations

In [ ]:
pose2d.figures["joint_trace"]


In [ ]:
global_pose.figures["global_pose"]


## 6) Build forces with semantic body targeting

You can target semantic segments like `right_foot`, `left_hand`, `pelvis`, or pass an explicit OpenSim body name.


In [ ]:
force_set = mm.ForceSet([
    mm.ExternalForce.constant(
        name="right_grf",
        target="right_foot",     # resolves to calcn_r in the GATMA preset/body map
        magnitude=900.0,
        direction=(0.0, 1.0, 0.0),
        point="right_ankle",
    ),
    mm.ExternalForce.constant(
        name="left_hand_load",
        target="left_hand",      # resolves to hand_l
        magnitude=75.0,
        direction=(0.0, -1.0, 0.0),
        point="left_wrist",
    ),
])

forces = mm.forces.build(
    trial,
    global_pose=global_pose,
    force_set=force_set,
)

forces.tables["forces_long"].head()


In [ ]:
forces.tables["mapping"]


## 7) Run GATMA-aligned OpenSim IK

This writes both OpenSim-native artifacts and CSV copies when available.

In [ ]:
ik = mm.opensim.run_ik(
    trial,
    global_pose=global_pose,
    model_path=str(OPENSIM_MODEL_PATH),
    config=config.opensim,
    output_dir=OUTPUT_DIR / "opensim_ik",
)

ik.tables["summary"]


In [ ]:
ik.tables["coordinates"].head()


In [ ]:
ik.artifacts


## 8) Run inverse dynamics with the same force set

In [ ]:
id_config = mm.OpenSimConfig(
    model_path=str(OPENSIM_MODEL_PATH),
    model_preset="gatma_exact",
    run_scale=True,
    run_ik=True,
    run_id=True,
    export_marker_trc=True,
    export_landmark_trc=True,
    write_csv_copies=True,
)

id_result = mm.opensim.run_id(
    trial,
    global_pose=global_pose,
    forces=forces,
    model_path=str(OPENSIM_MODEL_PATH),
    config=id_config,
    output_dir=OUTPUT_DIR / "opensim_id",
)

id_result.tables["summary"]


In [ ]:
id_result.tables["generalized_forces"].head()


In [ ]:
id_result.artifacts


## 9) Full wrapper version

The wrapper lets you define the whole pipeline up front, then run it in one call.

In [ ]:
pipeline = mm.FullPipeline(
    config=config,
    stages=mm.PipelineStages(
        pose2d=True,
        world3d=True,
        pnp=True,
        global_pose=True,
        forces=True,
        ik=True,
        id=False,
    ),
)

bundle = pipeline.run(
    VIDEO_PATH,
    output_dir=OUTPUT_DIR / "full_pipeline",
    force_set=force_set,
)

bundle.available_stages()


In [ ]:
bundle.ik.artifacts if bundle.ik is not None else {}


## 10) Expected outputs

When OpenSim stages run, look for artifacts such as:

- `*_global.trc` or `*_global_markers.trc`
- TRC CSV copies
- `*_Setup_Scale.xml`
- `*_scaled.osim`
- `*_ik.mot`
- `*_ik.csv`
- `*_external_loads.sto`
- `*_external_loads.csv`
- `*_ExternalLoads.xml`
- `*_id.sto`
- `*_id.csv`
